# NDLS → CNB Railway Delay Prediction — Model Training

This notebook trains and compares multiple regression models to predict
**`delay_next_station`** (delay in minutes at the next stop) for trains on the
New Delhi (NDLS) → Kanpur Central (CNB) corridor, using
`new_final_master_data.csv` produced by the data pipeline in this repo.

This is **time-series data**: each row is a (train, station, date) record with
lag/rolling delay features. We therefore:
- split train/test **by date** (chronological split, not random/shuffled),
- avoid any leakage from future rows into past rows (lag/rolling features in
  the source data were already computed leak-safe),
- evaluate with MAE / RMSE / R² on the held-out future time period.

We train several model families (linear baseline, tree ensembles, gradient
boosting, and an LSTM sequence model) and dump the best one at the end.

## How to use in Google Colab
1. Upload `new_final_master_data.csv` (from `data/processed/`) when prompted
   in the **Load data** cell below — or mount Google Drive and adjust the path.
2. Run all cells (Runtime → Run all).
3. The best model is saved to `best_delay_model.pkl` (and offered for download).


## 1. Setup — install & import dependencies

In [ ]:
!pip -q install lightgbm xgboost joblib


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import xgboost as xgb
import lightgbm as lgb

pd.set_option("display.max_columns", 50)


## 2. Load data

In Colab, run the cell below and upload `new_final_master_data.csv`
(found in `data/processed/` of the pipeline repo). If running locally /
with Drive mounted, just change `DATA_PATH` instead.


In [ ]:
DATA_PATH = "new_final_master_data.csv"

try:
    from google.colab import files
    import os
    if not os.path.exists(DATA_PATH):
        print("Please upload new_final_master_data.csv")
        uploaded = files.upload()
        DATA_PATH = list(uploaded.keys())[0]
except ImportError:
    pass  # not running in Colab; assume DATA_PATH already points to the file

df = pd.read_csv(DATA_PATH, dtype={"train_number": str})
df["date"] = pd.to_datetime(df["date"])
print(df.shape)
df.head()


## 3. Quick EDA

In [ ]:
print(df["delay_next_station"].describe())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df["delay_next_station"].dropna().clip(upper=120).hist(bins=50, ax=axes[0])
axes[0].set_title("Distribution of delay_next_station (clipped at 120 min)")
axes[0].set_xlabel("minutes")

df.groupby("station_code")["current_delay"].mean().sort_values().plot(
    kind="barh", ax=axes[1], title="Mean current_delay by station")
plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df.groupby("hour_of_day")["current_delay"].mean().plot(ax=axes[0], title="Mean delay by hour_of_day")
df.groupby("is_weekend")["current_delay"].mean().plot(kind="bar", ax=axes[1], title="Mean delay: weekday vs weekend")
plt.tight_layout()
plt.show()


## 4. Preprocessing

- Drop rows where the **target** (`delay_next_station`) is missing (terminus
  stations have no "next station").
- Select numeric + categorical features. Categorical features
  (`station_code`, `train_type`, `corridor_direction`, `calibration_source`)
  are one-hot encoded.
- Remaining NaNs (weather/infrastructure columns are only populated for the
  6 corridor stations) are filled with column medians — tree models handle
  this fine, and we add an `is_corridor_station` indicator so models can
  learn the missingness pattern itself.


In [ ]:
data = df.dropna(subset=["delay_next_station"]).copy()

# indicator for "weather/infra features are populated" (corridor stations only)
data["is_corridor_station"] = data["track_count"].notna().astype(int)

numeric_features = [
    "current_delay", "previous_station_delay",
    "train_average_delay", "station_average_delay",
    "delay_lag_1", "delay_lag_7", "delay_rolling_mean_7", "delay_rolling_mean_30",
    "priority", "hour_of_day", "day_of_week", "month",
    "is_weekend", "is_special_day",
    "rainfall", "temperature", "humidity",
    "track_count", "platform_count", "electrified", "max_speed_kmh",
    "congestion_score", "is_corridor_station", "is_imputed",
    "station_sequence",
]
categorical_features = ["station_code", "train_type", "corridor_direction", "calibration_source"]

target = "delay_next_station"

# median-fill numeric NaNs (mostly weather/infra outside the 6 corridor stations)
for c in numeric_features:
    if data[c].isna().any():
        data[c] = data[c].fillna(data[c].median())

X = data[numeric_features + categorical_features]
y = data[target]
print(X.shape, y.shape)


## 5. Time-based train/test split

Because this is time-series data, we **sort by date** and use the most recent
~15% of dates as the held-out test set — simulating "train on history,
predict the future".


In [ ]:
data_sorted = data.sort_values("date")
cutoff = data_sorted["date"].quantile(0.85)
print("Train/test cutoff date:", cutoff.date())

train_mask = data_sorted["date"] <= cutoff
test_mask = ~train_mask

X_train = data_sorted.loc[train_mask, numeric_features + categorical_features]
y_train = data_sorted.loc[train_mask, target]
X_test = data_sorted.loc[test_mask, numeric_features + categorical_features]
y_test = data_sorted.loc[test_mask, target]

print(f"Train rows: {len(X_train)}  ({X_train.index.size / len(data_sorted):.1%})")
print(f"Test rows:  {len(X_test)}  ({X_test.index.size / len(data_sorted):.1%})")


## 6. Preprocessing pipeline (shared by all models)

One-hot encode the categorical columns; numeric columns pass through
unchanged (tree/boosting models don't need scaling; the linear baseline
benefits from it too in this case since features are roughly comparable
delay/time scales, but we keep it simple and consistent across models for a
fair comparison).


In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
    ],
    remainder="passthrough",
)


## 7. Train multiple models

We train and compare:
1. **Ridge Regression** — linear baseline
2. **Random Forest Regressor** — bagged trees, robust to non-linearity
3. **Gradient Boosting Regressor** (sklearn) — boosted trees
4. **XGBoost Regressor** — optimized gradient boosting
5. **LightGBM Regressor** — fast gradient boosting, handles categoricals well

Each is wrapped in a `Pipeline` with the shared preprocessor so they're
evaluated on identical feature representations.


In [ ]:
models = {
    "Ridge": Ridge(alpha=1.0, random_state=42),
    "RandomForest": RandomForestRegressor(
        n_estimators=200, max_depth=14, n_jobs=-1, random_state=42),
    "GradientBoosting": GradientBoostingRegressor(
        n_estimators=200, max_depth=4, learning_rate=0.05, random_state=42),
    "XGBoost": xgb.XGBRegressor(
        n_estimators=300, max_depth=6, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8, n_jobs=-1, random_state=42),
    "LightGBM": lgb.LGBMRegressor(
        n_estimators=300, max_depth=-1, num_leaves=63, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8, n_jobs=-1, random_state=42),
}

results = {}
fitted_pipelines = {}

for name, model in models.items():
    pipe = Pipeline([("prep", preprocessor), ("model", model)])
    pipe.fit(X_train, y_train)
    preds = pipe.predict(X_test)

    mae = mean_absolute_error(y_test, preds)
    rmse = mean_squared_error(y_test, preds, squared=False)
    r2 = r2_score(y_test, preds)

    results[name] = {"MAE": mae, "RMSE": rmse, "R2": r2}
    fitted_pipelines[name] = pipe
    print(f"{name:18s} MAE={mae:6.2f}  RMSE={rmse:6.2f}  R2={r2:6.3f}")


## 8. Optional: LSTM sequence model

Tree/linear models above treat each (train, station, date) row independently
(using lag/rolling features as a proxy for history). As a bonus, this section
builds a small **LSTM** that consumes the last `SEQ_LEN` days of
`delay_minutes` for each (train, station) pair to predict the next day's
`current_delay` — a genuinely sequential view of the same data.

This cell requires `tensorflow` (pre-installed in Colab). It's optional and
slower than the tree models above; skip it if you only need the tabular
models.


In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models as keras_models

SEQ_LEN = 14  # days of history per (train, station)

seq_df = data_sorted[["train_number", "station_code", "date", "current_delay"]].copy()
seq_df = seq_df.sort_values(["train_number", "station_code", "date"])

def make_sequences(group, seq_len=SEQ_LEN):
    vals = group["current_delay"].values
    X_seq, y_seq, dates = [], [], []
    for i in range(len(vals) - seq_len):
        X_seq.append(vals[i:i + seq_len])
        y_seq.append(vals[i + seq_len])
        dates.append(group["date"].values[i + seq_len])
    return X_seq, y_seq, dates

all_X, all_y, all_dates = [], [], []
for _, g in seq_df.groupby(["train_number", "station_code"]):
    xs, ys, ds = make_sequences(g)
    all_X.extend(xs)
    all_y.extend(ys)
    all_dates.extend(ds)

all_X = np.array(all_X, dtype="float32")[..., np.newaxis]  # (samples, seq_len, 1)
all_y = np.array(all_y, dtype="float32")
all_dates = pd.to_datetime(all_dates)

lstm_cutoff = pd.Series(all_dates).quantile(0.85)
train_idx = all_dates <= lstm_cutoff
test_idx = ~train_idx

X_train_lstm, y_train_lstm = all_X[train_idx], all_y[train_idx]
X_test_lstm, y_test_lstm = all_X[test_idx], all_y[test_idx]
print(X_train_lstm.shape, X_test_lstm.shape)

lstm = keras_models.Sequential([
    layers.Input(shape=(SEQ_LEN, 1)),
    layers.LSTM(32, return_sequences=False),
    layers.Dense(16, activation="relu"),
    layers.Dense(1),
])
lstm.compile(optimizer="adam", loss="mae")
lstm.fit(X_train_lstm, y_train_lstm, validation_split=0.1, epochs=10, batch_size=256, verbose=1)

lstm_preds = lstm.predict(X_test_lstm).flatten()
mae = mean_absolute_error(y_test_lstm, lstm_preds)
rmse = mean_squared_error(y_test_lstm, lstm_preds, squared=False)
r2 = r2_score(y_test_lstm, lstm_preds)
results["LSTM"] = {"MAE": mae, "RMSE": rmse, "R2": r2}
print(f"{'LSTM':18s} MAE={mae:6.2f}  RMSE={rmse:6.2f}  R2={r2:6.3f}")


## 9. Compare models & pick the best


In [ ]:
results_df = pd.DataFrame(results).T.sort_values("MAE")
print(results_df)

results_df["MAE"].plot(kind="barh", figsize=(8, 4), title="Test MAE by model (lower is better)")
plt.xlabel("MAE (minutes)")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()


## 10. Save the best model

Picks the model with the lowest test MAE among the **tabular** pipelines
(Ridge / RandomForest / GradientBoosting / XGBoost / LightGBM — these are
saved as a single sklearn `Pipeline` including preprocessing, so they can be
applied directly to a raw feature DataFrame). The LSTM, if trained, is
reported for comparison but saved separately since it has a different input
format (delay sequences, not tabular rows).


In [ ]:
tabular_results = {k: v for k, v in results.items() if k in fitted_pipelines}
best_name = min(tabular_results, key=lambda k: tabular_results[k]["MAE"])
best_pipeline = fitted_pipelines[best_name]

print(f"Best tabular model: {best_name}")
print(tabular_results[best_name])

joblib.dump(best_pipeline, "best_delay_model.pkl")
joblib.dump({"numeric_features": numeric_features, "categorical_features": categorical_features,
              "target": target, "model_name": best_name},
             "best_delay_model_metadata.pkl")

if "LSTM" in results:
    lstm.save("lstm_delay_model.keras")

print("Saved best_delay_model.pkl (+ metadata)")


In [ ]:
# In Colab, download the saved model files:
try:
    from google.colab import files
    files.download("best_delay_model.pkl")
    files.download("best_delay_model_metadata.pkl")
except ImportError:
    pass
